In [4]:
# Cell 1 — Imports
import os
import time
import joblib
import numpy as np
import pandas as pd
import fastf1

from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)

In [5]:
# Cell 2 — Paths and config
BASE_DIR   = os.path.dirname(os.getcwd())
DATA_PATH  = os.path.join(BASE_DIR, "data",   "f1_dataset_clean.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "model_v8.pkl")
CACHE_PATH = os.path.join(BASE_DIR, "cache")

os.makedirs(CACHE_PATH,                  exist_ok=True)
os.makedirs(os.path.dirname(DATA_PATH),  exist_ok=True)
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

fastf1.Cache.enable_cache(CACHE_PATH)

HISTORICAL_YEARS = [2023, 2024, 2025, 2026]
DECAY_FACTOR     = 0.38   # aggressive recency bias — 2026 is a regulation-change year
TARGET           = "Podium"

In [6]:
# Cell 3 — Track type lookup
TRACK_TYPE = {
    # Street circuits
    "Jeddah":        "street",
    "Baku":          "street",
    "Miami":         "street",
    "Monaco":        "street",
    "Marina Bay":    "street",
    "Las Vegas":     "street",
    "Melbourne":     "street",
    "Miami Gardens": "street",
    "Madrid":        "street",
    # Permanent circuits
    "Sakhir":            "permanent",
    "Barcelona":         "permanent",
    "Montréal":          "permanent",
    "Spielberg":         "permanent",
    "Silverstone":       "permanent",
    "Budapest":          "permanent",
    "Spa-Francorchamps": "permanent",
    "Zandvoort":         "permanent",
    "Monza":             "permanent",
    "Suzuka":            "permanent",
    "Lusail":            "permanent",
    "Austin":            "permanent",
    "Mexico City":       "permanent",
    "São Paulo":         "permanent",
    "Yas Island":        "permanent",
    "Shanghai":          "permanent",
    "Imola":             "permanent",
}

In [46]:
# Cell 4 — Fetch all race + qualifying data
def fetch_round(year, round_num):
    try:
        session_r = fastf1.get_session(year, round_num, "R")
        session_r.load(laps=False, telemetry=False, weather=True, messages=False)  # ← weather=True
        race = session_r.results[["FullName", "TeamName", "GridPosition", "Position", "Status"]].copy()
        race["Year"]     = year
        race["Round"]    = round_num
        race["Location"] = session_r.event["Location"]

        # RainFlag — 1 if any rainfall detected during race
        try:
            rain = session_r.weather_data["Rainfall"].any()
            race["RainFlag"] = int(rain)
        except Exception:
            race["RainFlag"] = 0

    except Exception as e:
        print(f"  ✗ Race failed {year} R{round_num}: {e}")
        return None

    try:
        session_q = fastf1.get_session(year, round_num, "Q")
        session_q.load(laps=False, telemetry=False, weather=False, messages=False)
        q = session_q.results[["FullName", "Q1", "Q2", "Q3"]].copy()
        q["BestQualiTime"] = q[["Q1", "Q2", "Q3"]].min(axis=1).dt.total_seconds()
        q = q[["FullName", "BestQualiTime"]]
    except Exception as e:
        print(f"  ✗ Quali failed {year} R{round_num}: {e}")
        q = pd.DataFrame(columns=["FullName", "BestQualiTime"])

    df = race.merge(q, on="FullName", how="left")
    df["Podium"]    = (df["Position"] <= 3).astype(int)
    df["TrackType"] = df["Location"].map(TRACK_TYPE)

    if df["TrackType"].isnull().any():
        unknown = df[df["TrackType"].isnull()]["Location"].unique()
        print(f"  ⚠ Unknown locations — add to TRACK_TYPE: {unknown}")

    time.sleep(1)
    return df

frames = []
for year in HISTORICAL_YEARS:
    print(f"\n── {year} ──────────────────────")
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    for round_num in schedule["RoundNumber"].tolist():
        df = fetch_round(year, round_num)
        if df is not None and len(df) > 0:
            frames.append(df)

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows : {len(df_raw)}")
print(f"Years      : {sorted(df_raw['Year'].unique())}")
print(f"Rounds/year: {df_raw.groupby('Year')['Round'].nunique().to_dict()}")


── 2023 ──────────────────────


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INF


── 2024 ──────────────────────


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 


── 2025 ──────────────────────


...

  ✗ Race failed 2026 R19: any API: 500 calls/h


core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  ✗ Race failed 2026 R20: any API: 500 calls/h


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  ✗ Race failed 2026 R21: any API: 500 calls/h
  ✗ Race failed 2026 R22: any API: 500 calls/h

Total rows : 1486
Years      : [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Rounds/year: {2023: 22, 2024: 24, 2025: 24, 2026: 4}


In [47]:
# Cell 5 — Clean data
def clean(df):
    df = df.copy()
    df = df.dropna(subset=["Position"])
    df["Position"]     = df["Position"].astype(int)
    df["GridPosition"] = df["GridPosition"].fillna(20).astype(int)
    worst_per_round    = df.groupby(["Year", "Round"])["BestQualiTime"].transform("max")
    df["BestQualiTime"] = df["BestQualiTime"].fillna(worst_per_round + 5.0)
    return df.sort_values(["Year", "Round", "Position"]).reset_index(drop=True)

df_clean = clean(df_raw)
print(f"Rows: {len(df_clean)}  |  Missing values: {df_clean.isnull().sum().sum()}")

Rows: 1485  |  Missing values: 0


In [48]:
# Cell 6 — Feature engineering
def other_driver_mean(df, by, column):
    counts    = df.groupby(by)[column].transform("count")
    totals    = df.groupby(by)[column].transform("sum")
    peer_mean = (totals - df[column]) / (counts - 1)
    return peer_mean.where(counts > 1, df[column])


def engineer_features(df):
    df = df.copy()
    round_key      = ["Year", "Round"]
    team_round_key = ["Year", "Round", "TeamName"]

    # Qualifying
    df["QualiGapNormalized"]  = df.groupby(round_key)["BestQualiTime"].transform(lambda x: (x - x.min()) / x.min() * 100)
    df["GridPositionSquared"] = df["GridPosition"] ** 2

    # Teammate comparison
    teammate_grid          = other_driver_mean(df, team_round_key, "GridPosition")
    teammate_quali         = other_driver_mean(df, team_round_key, "BestQualiTime")
    df["TeammateGridDelta"] = df["GridPosition"] - teammate_grid
    df["TeammateQualiGap"]  = df["BestQualiTime"] - teammate_quali

    # Rolling driver form (leakage-safe: shift(1) excludes current race)
    df["PositionGain"] = df["GridPosition"] - df["Position"]
    df = df.sort_values(["FullName", "Year", "Round"])

    df["AvgPositionGainLast3"] = df.groupby(["FullName", "Year"])["PositionGain"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(0.0)
    df["FinishStdLast5"]       = df.groupby(["FullName", "Year"])["Position"].transform(lambda x: x.shift(1).rolling(5, min_periods=2).std()).fillna(5.0)
    df["AvgFinishLast3"]       = df.groupby(["FullName", "Year"])["Position"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(10.0)
    df["PodiumRateLast5"]      = df.groupby(["FullName", "Year"])["Podium"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0.15)
    df["DNF"]                  = df["Status"].isin(["Retired", "Accident", "Collision damage", "Undertray", "Withdrew"]).astype(int)
    df["DNFRateLast5"]         = df.groupby(["FullName", "Year"])["DNF"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0.1)

    teammate_position      = other_driver_mean(df, team_round_key, "Position")
    df["BeatTeammate"]     = (df["Position"] < teammate_position).astype(int)
    df["BeatTeammateRate"] = df.groupby(["FullName", "Year"])["BeatTeammate"].transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean()).fillna(0.5)

    # Rolling constructor form
    team_round = (
        df.groupby(["TeamName", "Year", "Round"], as_index=False)
        .agg(TeamPodiumCurrent=("Podium", "mean"), TeamAvgFinishCurrent=("Position", "mean"))
        .sort_values(["TeamName", "Year", "Round"])
    )
    team_round["ConstructorPodiumRate"] = team_round.groupby(["TeamName", "Year"])["TeamPodiumCurrent"].transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean()).fillna(0.1)
    team_round["ConstructorAvgFinish"]  = team_round.groupby(["TeamName", "Year"])["TeamAvgFinishCurrent"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(10.0)

    df = df.merge(team_round[["TeamName", "Year", "Round", "ConstructorPodiumRate", "ConstructorAvgFinish"]], on=["TeamName", "Year", "Round"], how="left")
    df = df.sort_values(["Year", "Round", "GridPosition"]).reset_index(drop=True)
    df = df.drop(columns=["PositionGain", "DNF", "BeatTeammate"])
    return df


df_features = engineer_features(df_clean)
df_features.to_csv(DATA_PATH, index=False)
print(f"Saved {len(df_features)} rows → {DATA_PATH}")

Saved 1485 rows → d:\F1\data\f1_dataset_clean.csv


In [75]:
# Cell 7 — Track history features
def add_track_history(df):
    df = df.sort_values(["Year", "Round", "GridPosition"]).copy()

    df["DriverCircuitAvgFinish"]      = df.groupby(["FullName", "Location"])["Position"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(10.0)
    df["DriverCircuitPodiumRate"]     = df.groupby(["FullName", "Location"])["Podium"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(0.15)
    df["ConstructorCircuitAvgFinish"] = df.groupby(["TeamName", "Location"])["Position"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(10.0)
    df["ConstructorCircuitPodiumRate"]= df.groupby(["TeamName", "Location"])["Podium"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(0.15)

    race_stats = []
    for (year, rnd, location), grp in df.groupby(["Year", "Round", "Location"]):
        pole_row = grp.loc[grp["GridPosition"].idxmin()]
        race_stats.append({
            "Year": year, "Round": rnd, "Location": location,
            "CurrentAbsPositionChange":    (grp["GridPosition"] - grp["Position"]).abs().mean(),
            "CurrentPolePodium":           int(pole_row["Podium"] == 1),
            "CurrentPodiumOutsideGridTop3": int(((grp["GridPosition"] > 3) & (grp["Podium"] == 1)).any()),
        })

    rs = pd.DataFrame(race_stats).sort_values(["Location", "Year", "Round"])
    rs["TrackOvertakingDifficulty"] = -rs.groupby("Location")["CurrentAbsPositionChange"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(-3.0)
    rs["TrackPolePodiumRate"]       =  rs.groupby("Location")["CurrentPolePodium"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(0.65)
    rs["TrackSafetyCarProxy"]       =  rs.groupby("Location")["CurrentPodiumOutsideGridTop3"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean()).fillna(0.30)

    return df.merge(rs[["Year", "Round", "Location", "TrackOvertakingDifficulty", "TrackPolePodiumRate", "TrackSafetyCarProxy"]], on=["Year", "Round", "Location"], how="left")


df_v8 = add_track_history(df_features)
print(f"v8 feature frame: {df_v8.shape}")

v8 feature frame: (1485, 31)


In [76]:
POINTS_MAP = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10,
              6: 8,  7: 6,  8: 4,  9: 2, 10: 1}

df_v8["PointsEarned"] = df_v8["Position"].map(POINTS_MAP).fillna(0)

df_v8 = df_v8.sort_values(["Year", "Round"])

# Cumsum first, then shift within each driver-year group
df_v8["CumPoints"] = df_v8.groupby(["Year", "FullName"])["PointsEarned"].cumsum()
df_v8["ChampionshipPoints"] = df_v8.groupby(["Year", "FullName"])["CumPoints"].shift(1).fillna(0)

df_v8.drop(columns=["PointsEarned", "CumPoints"], inplace=True)

print(df_v8[["FullName", "Year", "Round", "ChampionshipPoints"]].head(20))

           FullName  Year  Round  ChampionshipPoints
0    Max Verstappen  2023      1                 0.0
1      Sergio Perez  2023      1                 0.0
2   Charles Leclerc  2023      1                 0.0
3      Carlos Sainz  2023      1                 0.0
4   Fernando Alonso  2023      1                 0.0
5    George Russell  2023      1                 0.0
6    Lewis Hamilton  2023      1                 0.0
7      Lance Stroll  2023      1                 0.0
8      Esteban Ocon  2023      1                 0.0
9   Nico Hulkenberg  2023      1                 0.0
10     Lando Norris  2023      1                 0.0
11  Valtteri Bottas  2023      1                 0.0
12      Guanyu Zhou  2023      1                 0.0
13     Yuki Tsunoda  2023      1                 0.0
14  Alexander Albon  2023      1                 0.0
15   Logan Sargeant  2023      1                 0.0
16  Kevin Magnussen  2023      1                 0.0
17    Oscar Piastri  2023      1              

In [77]:
df_v8 = df_v8.sort_values(["Year", "Round"])

# Average finish per constructor per round
constructor_round_avg = (
    df_v8.groupby(["Year", "Round", "TeamName"])["Position"]
    .mean()
    .reset_index()
    .rename(columns={"Position": "ConstructorRoundAvgFinish"})
    .drop_duplicates(subset=["Year", "Round", "TeamName"])
)

# For each constructor-year, compute first 3 races avg and rolling last 3 avg
constructor_round_avg = constructor_round_avg.sort_values(["Year", "TeamName", "Round"])

constructor_round_avg["ConstructorFirst3Avg"] = (
    constructor_round_avg.groupby(["Year", "TeamName"])["ConstructorRoundAvgFinish"]
    .transform(lambda x: x.iloc[:3].mean())
)

constructor_round_avg["ConstructorLast3Avg"] = (
    constructor_round_avg.groupby(["Year", "TeamName"])["ConstructorRoundAvgFinish"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

constructor_round_avg["ConstructorDevelopmentRate"] = (
    constructor_round_avg["ConstructorLast3Avg"] - constructor_round_avg["ConstructorFirst3Avg"]
)

# Merge back
df_v8 = df_v8.merge(
    constructor_round_avg[["Year", "Round", "TeamName", "ConstructorDevelopmentRate"]],
    on=["Year", "Round", "TeamName"], how="left"
)

df_v8["ConstructorDevelopmentRate"] = df_v8["ConstructorDevelopmentRate"].fillna(0)

print(df_v8[["FullName", "TeamName", "Year", "Round", "ConstructorDevelopmentRate"]]
      .sort_values(["Year", "Round"])
      .head(30))

           FullName         TeamName  Year  Round  ConstructorDevelopmentRate
0    Max Verstappen  Red Bull Racing  2023      1                    0.000000
1      Sergio Perez  Red Bull Racing  2023      1                    0.000000
2   Charles Leclerc          Ferrari  2023      1                    0.000000
3      Carlos Sainz          Ferrari  2023      1                    0.000000
4   Fernando Alonso     Aston Martin  2023      1                    0.000000
5    George Russell         Mercedes  2023      1                    0.000000
6    Lewis Hamilton         Mercedes  2023      1                    0.000000
7      Lance Stroll     Aston Martin  2023      1                    0.000000
8      Esteban Ocon           Alpine  2023      1                    0.000000
9   Nico Hulkenberg     Haas F1 Team  2023      1                    0.000000
10     Lando Norris          McLaren  2023      1                    0.000000
11  Valtteri Bottas       Alfa Romeo  2023      1               

In [78]:
# Current season avg finish — excludes historical data, only this year's races
df_v8 = df_v8.sort_values(["Year", "Round"])

df_v8["CurrentSeasonAvgFinish"] = (
    df_v8.groupby(["Year", "FullName"])["Position"]
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(df_v8["Position"].median())  # fallback for R1 where no prior data exists
)

print(df_v8[["FullName", "Year", "Round", "Position", "CurrentSeasonAvgFinish"]]
      .sort_values(["Year", "Round"])
      .head(30))

           FullName  Year  Round  Position  CurrentSeasonAvgFinish
0    Max Verstappen  2023      1         1                    11.0
1      Sergio Perez  2023      1         2                    11.0
2   Charles Leclerc  2023      1        19                    11.0
3      Carlos Sainz  2023      1         4                    11.0
4   Fernando Alonso  2023      1         3                    11.0
5    George Russell  2023      1         7                    11.0
6    Lewis Hamilton  2023      1         5                    11.0
7      Lance Stroll  2023      1         6                    11.0
8      Esteban Ocon  2023      1        18                    11.0
9   Nico Hulkenberg  2023      1        15                    11.0
10     Lando Norris  2023      1        17                    11.0
11  Valtteri Bottas  2023      1         8                    11.0
12      Guanyu Zhou  2023      1        16                    11.0
13     Yuki Tsunoda  2023      1        11                    

In [84]:
# Cell 8 — Define features, build model frame, split
FEATURE_COLS = [
    # Qualifying / grid
    "GridPosition",
    "QualiGapNormalized",
    # Driver race craft
    "AvgPositionGainLast3",
    "FinishStdLast5",
    "DNFRateLast5",
    # Driver form
    "AvgFinishLast3",
    "PodiumRateLast5",
    "BeatTeammateRate",
    "CurrentSeasonAvgFinish",
    # Constructor form
    "ConstructorPodiumRate",
    "ConstructorAvgFinish",
    "ConstructorDevelopmentRate",
    # Track type
    "TrackType_street",
    "TrackType_permanent",
    # Rain
    "RainFlag",
]

df_model = pd.get_dummies(df_v8, columns=["TrackType"], dtype=int)
df_model["Winner"] = (df_model["Position"] == 1).astype(int)
for col in ["TrackType_street", "TrackType_permanent"]:
    if col not in df_model.columns:
        df_model[col] = 0
for col in FEATURE_COLS:
    if col not in df_model.columns:
        df_model[col] = 0.0
    df_model[col] = df_model[col].fillna(df_model[col].median())

# tune_df / val_df used only for Optuna HPO — no test data seen during tuning
tune_df  = df_model[df_model["Year"] < 2025].copy()   # 2023–2024 — HPO train
val_df   = df_model[df_model["Year"] == 2025].copy()  # 2025     — HPO validation
train_df = df_model[df_model["Year"] < 2026].copy()   # 2023–2025 — final train
test_df  = df_model[df_model["Year"] == 2026].copy()  # 2026     — clean holdout

print(f"Tune: {len(tune_df)}  Val: {len(val_df)}  Train: {len(train_df)}  Test: {len(test_df)}")
print(f"Podium rate — tune: {tune_df[TARGET].mean():.3f}  val: {val_df[TARGET].mean():.3f}  test: {test_df[TARGET].mean():.3f}")

Tune: 918  Val: 479  Train: 1397  Test: 88
Podium rate — tune: 0.150  val: 0.150  test: 0.136


In [85]:
# Cell 9 — Evaluation helpers
def top3_hits(eval_df, score_col, largest=True):
    correct, total = 0, 0
    for (yr, rnd), grp in eval_df.groupby(["Year", "Round"]):
        ranked      = grp.nlargest(3, score_col) if largest else grp.nsmallest(3, score_col)
        pred_top3   = set(ranked["FullName"])
        actual_top3 = set(grp[grp[TARGET] == 1]["FullName"])
        correct += len(pred_top3 & actual_top3)
        total   += min(3, len(actual_top3))
    return correct, total, correct / total if total else np.nan


def evaluate(eval_df, proba, label, combined_proba=None):
    eval_df = eval_df.copy()
    eval_df["proba"] = proba
    # Use combined_proba for top-3 ranking if provided, else fall back to proba
    eval_df["ranking_score"] = combined_proba if combined_proba is not None else proba
    y_true = eval_df[TARGET]

    print(f"\n── {label} ──")
    print(f"ROC AUC:           {roc_auc_score(y_true, proba):.4f}")
    print(f"Average Precision: {average_precision_score(y_true, proba):.4f}")
    print(f"Brier Score:       {brier_score_loss(y_true, proba):.4f}")

    m_c, m_t, m_r = top3_hits(eval_df, "ranking_score", largest=True)
    g_c, g_t, g_r = top3_hits(eval_df, "GridPosition",  largest=False)
    print(f"Model Top-3:       {m_c}/{m_t} ({m_r:.1%})")
    print(f"Grid baseline:     {g_c}/{g_t} ({g_r:.1%})")
    print(f"Edge vs grid:      {m_r - g_r:+.1%}")

    print("\nPer-round breakdown:")
    for (yr, rnd), grp in eval_df.groupby(["Year", "Round"]):
        pred   = set(grp.nlargest(3, "ranking_score")["FullName"])
        actual = set(grp[grp[TARGET] == 1]["FullName"])
        grid   = set(grp.nsmallest(3, "GridPosition")["FullName"])
        print(f"  {yr} R{rnd:2d}: model {len(pred & actual)}/3  grid {len(grid & actual)}/3  |  predicted: {sorted(pred)}")

In [86]:
# Cell 10 — Hyperparameter tuning + train + save
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Tune on 2023-2024, validate on 2025 — test (2026) never seen during HPO
tune_weights = (DECAY_FACTOR ** (2024 - tune_df["Year"])).values

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 3, 7),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 30),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight":  trial.suggest_float("scale_pos_weight", 4, 10),
        "random_state": 42,
        "verbose": -1,
    }
    model = CalibratedClassifierCV(LGBMClassifier(**params), cv=3, method="isotonic")
    model.fit(tune_df[FEATURE_COLS], tune_df[TARGET], sample_weight=tune_weights)
    proba = model.predict_proba(val_df[FEATURE_COLS])[:, 1]
    return average_precision_score(val_df[TARGET], proba)

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest AP (val):  {study.best_value:.4f}")
print(f"Best params:    {study.best_params}")

# Retrain on full 2023-2025 with best params
best_params  = study.best_params | {"random_state": 42, "verbose": -1}
train_weights = (DECAY_FACTOR ** (2025 - train_df["Year"])).values

podium_model = CalibratedClassifierCV(LGBMClassifier(**best_params), cv=3, method="isotonic")
podium_model.fit(train_df[FEATURE_COLS], train_df[TARGET], sample_weight=train_weights)

winner_params = best_params.copy()
winner_params["scale_pos_weight"] = (train_df["Winner"] == 0).sum() / (train_df["Winner"] == 1).sum()

winner_model = CalibratedClassifierCV(LGBMClassifier(**winner_params), cv=3, method="isotonic")
winner_model.fit(train_df[FEATURE_COLS], train_df["Winner"], sample_weight=train_weights)

podium_proba = podium_model.predict_proba(test_df[FEATURE_COLS])[:, 1]
winner_proba = winner_model.predict_proba(test_df[FEATURE_COLS])[:, 1]
combined_proba = 0.6 * podium_proba + 0.4 * winner_proba

test_df = test_df.reset_index(drop=True)
test_df["PodiumProba"]   = podium_proba
test_df["WinnerProba"]   = winner_proba
test_df["CombinedScore"] = combined_proba

evaluate(test_df, podium_proba, "2026 holdout (clean)", combined_proba=combined_proba)

joblib.dump(podium_model, MODEL_PATH)
joblib.dump(winner_model, MODEL_PATH.replace("model_v8", "model_v8_winner"))
print(f"\nModels saved → {MODEL_PATH}")

Best trial: 64. Best value: 0.824298: 100%|██████████| 100/100 [00:25<00:00,  3.96it/s]



Best AP (val):  0.8243
Best params:    {'n_estimators': 554, 'max_depth': 3, 'learning_rate': 0.011048312218739881, 'num_leaves': 15, 'min_child_samples': 29, 'subsample': 0.7070589642174864, 'colsample_bytree': 0.7429137486847875, 'scale_pos_weight': 9.72604240924162}

── 2026 holdout (clean) ──
ROC AUC:           0.9380
Average Precision: 0.7127
Brier Score:       0.0649
Model Top-3:       8/12 (66.7%)
Grid baseline:     8/12 (66.7%)
Edge vs grid:      +0.0%

Per-round breakdown:
  2026 R 1: model 2/3  grid 2/3  |  predicted: ['George Russell', 'Isack Hadjar', 'Kimi Antonelli']
  2026 R 2: model 3/3  grid 3/3  |  predicted: ['George Russell', 'Kimi Antonelli', 'Lewis Hamilton']
  2026 R 3: model 2/3  grid 2/3  |  predicted: ['Charles Leclerc', 'George Russell', 'Kimi Antonelli']
  2026 R 4: model 1/3  grid 1/3  |  predicted: ['Charles Leclerc', 'Kimi Antonelli', 'Max Verstappen']

Models saved → d:\F1\models\model_v8.pkl


In [87]:
# Top-3 per race sorted by combined score
for race_id, group in test_df.groupby(["Year", "Round"]):
    top3 = group.nlargest(3, "CombinedScore")[["FullName", "CombinedScore", "WinnerProba", "PodiumProba"]]
    print(f"\n{race_id}")
    print(top3.to_string(index=False))


(np.int64(2026), np.int64(1))
      FullName  CombinedScore  WinnerProba  PodiumProba
George Russell       0.715165     0.734793     0.702080
Kimi Antonelli       0.352894     0.058968     0.548845
  Isack Hadjar       0.182760     0.000000     0.304601

(np.int64(2026), np.int64(2))
      FullName  CombinedScore  WinnerProba  PodiumProba
Kimi Antonelli       0.711308     0.610464     0.778537
George Russell       0.600596     0.293584     0.805271
Lewis Hamilton       0.348444     0.080960     0.526767

(np.int64(2026), np.int64(3))
       FullName  CombinedScore  WinnerProba  PodiumProba
 Kimi Antonelli       0.832598     0.664086     0.944939
 George Russell       0.663730     0.264494     0.929887
Charles Leclerc       0.288594     0.070139     0.434230

(np.int64(2026), np.int64(4))
       FullName  CombinedScore  WinnerProba  PodiumProba
 Kimi Antonelli       0.828011     0.664086     0.937295
Charles Leclerc       0.449803     0.143194     0.654209
 Max Verstappen       0.35025